# 05 — Exercises

Write the code yourself. Solutions are in `examples/` folder.

---
## Exercise 1 — Single ReAct agent with memory

**Task:**
- Create a `create_react_agent` with two tools:
  - `list_tables()` → returns `["site_metrics", "labor_events", "pipeline_runs"]`
  - `get_row_count(table_name: str)` → returns hardcoded dict lookup
- Enable `InMemorySaver`
- Run two turns on the same `thread_id`:
  - Turn 1: `"what tables do we have?"`
  - Turn 2: `"how many rows does labor_events have?"`
- Agent should NOT call `list_tables` again in turn 2

In [ ]:
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_ollama import ChatOllama
from langchain_core.tools import tool

llm = ChatOllama(model="llama3.2:3b")

# TODO: define list_tables tool

# TODO: define get_row_count tool

# TODO: create InMemorySaver

# TODO: create agent

config = {"configurable": {"thread_id": "ex1"}}

# TODO: turn 1 — invoke and print

# TODO: turn 2 — invoke and print

---
## Exercise 2 — StateGraph with conditional routing

**Task:**
- State: `{date: str, row_count: int, log: Annotated[list[str], operator.add]}`
- Nodes: `count_rows` (sets row_count=75000) → route → `full_load` OR `sample_load`
- If row_count > 50000 → full_load, else → sample_load
- Both load nodes append to log and go to END
- Print `result["log"]` at the end

In [ ]:
import operator
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    date: str
    row_count: int
    log: Annotated[list[str], operator.add]

# TODO: count_rows node

# TODO: full_load node

# TODO: sample_load node

# TODO: routing function

# TODO: build graph

# TODO: compile and invoke

# TODO: print result["log"]

---
## Exercise 3 — Two-agent supervisor

**Task:**
- `ingestion_agent` with tool: `read_from_s3(date: str)` → `"read 50000 rows"`
- `quality_agent` with tool: `run_dq(table: str)` → `"0 nulls found"`
- `supervisor_node` routes to `"ingestion"`, `"quality"`, or `"FINISH"` using `llm.with_structured_output(Router)`
- Worker nodes return `Command(goto="supervisor")`
- Memory enabled
- Test: `"ingest data for 2024-01-15 then check quality on site_metrics"`
- Print each supervisor routing decision

In [ ]:
from typing import Literal
from typing_extensions import TypedDict, Annotated
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.messages import AIMessage

llm = ChatOllama(model="llama3.2:3b")

# TODO: define tools

# TODO: create worker agents

# TODO: ingestion_node → Command(goto="supervisor")

# TODO: quality_node → Command(goto="supervisor")

# TODO: Router TypedDict

# TODO: supervisor_node

# TODO: build and compile graph

# graph.invoke({"messages": [{"role": "user", "content": "ingest data for 2024-01-15 then check quality on site_metrics"}]}, config)

---
## Exercise 4 — Parallel map-reduce

**Task:**
- State: `{regions: list[str], date: str, results: Annotated[list[dict], operator.add]}`
- `spawn_workers` → returns one `Send("process_region", ...)` per region
- `process_region` → returns `{"results": [{"region": x, "rows": <hardcoded>}]}`
  - Use: `{"na": 50000, "eu": 32000, "fe": 18000}`
- `aggregate` → prints total rows
- Run with `regions=["na", "eu", "fe"]`

In [ ]:
import operator
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.constants import Send

class OverallState(TypedDict):
    regions: list[str]
    date: str
    results: Annotated[list[dict], operator.add]

class RegionState(TypedDict):
    region: str
    date: str

# TODO: spawn_workers(state) → list[Send]

# TODO: process_region(state: RegionState) → dict

# TODO: aggregate(state: OverallState) → dict

# TODO: build graph with fan-out and fan-in

# TODO: compile and invoke

---
## Interview Questions — Write answers in the cells below

Answer first, then verify against the module.

**Q1:** What's the difference between `add_messages` reducer and a plain list in state? When do you need a reducer?

*Your answer:*

**Q2:** You have a 10-turn conversation. The user asks "what did I say at the start?" How does LangGraph know?

*Your answer:*

**Q3:** Your agent graph runs 50 nodes for a batch job. Halfway through it crashes. How do you resume from step 25?

*Your answer:*

**Q4:** You want the agent to remember a user's table preferences across different days/sessions. Checkpointer or store? Why?

*Your answer:*

**Q5:** Supervisor vs handoff — which would you use for a DE pipeline that needs audit logs? Why?

*Your answer:*